# SoundFont Visualization Notebook

This notebook probes a piano SoundFont across pitch and MIDI velocity.

It will:
1. render one pitch sweep per velocity
2. extract note-wise Bark dB and Ntot sone metrics
3. save a merged CSV
4. generate the final figure

Recommended environment:
- `mido`
- `pretty_midi`
- `pyfluidsynth` for `.sf2`
- `sfizz_render` for `.sfz`
- `torch` + `torchaudio`


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
from IPython.display import display

sys.path.insert(0, str(Path.cwd() / "src"))

from src.rendering.soundfont_visualization import run_probe_and_plot, load_soundfont_probe_csv, plot_soundfont_response_figure


In [ ]:
# Edit these paths for your machine.
instrument_path = Path('/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/YDP-GrandPiano-SF2-20160804/YDP-GrandPiano-20160804.sf2')
out_dir = Path('./figures/sf2_probe/YDP-GrandPiano')

# Typical piano range is 21..108.
# If you want the same x-axis style as your reference figure,
# you can switch to pitch_min=12.
probe_kwargs = dict(
    instrument_path=instrument_path,
    out_dir=out_dir,
    backend='auto',
    render_sr=44100,
    eval_sr=22050,
    pitch_min=21,
    pitch_max=108,
    velocity_min=10,
    velocity_max=110,
    velocity_step=2,
    highlight_velocity_step=10,
    note_duration_sec=2.2,
    analysis_duration_sec=2.0,
    inter_note_gap_sec=0.2,
    initial_silence_sec=0.1,
    final_tail_sec=1.0,
    fft_size=1024,
    frames_per_second=50.0,
    db_max=96.0,
    outer_ear='terhardt',
    keep_wav=False,
    keep_midi=True,
    overwrite=False,
)
probe_kwargs


In [ ]:
# This can take a long time for a dense velocity grid.
# The code is resume-friendly at the velocity level.
result = run_probe_and_plot(**probe_kwargs)
result


In [ ]:
df = load_soundfont_probe_csv(result['csv_path'])
display(df.head())
print('rows =', len(df))
print('figure =', result['figure_path'])


In [ ]:
# Re-plot from an existing CSV without re-rendering audio.
csv_path = Path(result['csv_path'])
df = load_soundfont_probe_csv(csv_path)
fig = plot_soundfont_response_figure(
    df,
    output_path=out_dir / 'replot_soundfont_response.png',
    instrument_name=instrument_path.stem,
    title=instrument_path.stem,
)
fig


## Notes

- `bark_peak_db_avg` is the peak over time of the mean Bark-band level.
- `ntot_peak_sone` is the peak total loudness in sones.
- The default layout uses one MIDI per velocity. This is much easier to resume.
- For a faster first test, try `velocity_step=10` and a smaller pitch range.
